## Silver Layer — Cleaned & Modeled Star Schema

This notebook transforms the raw bronze tables into a proper **star schema** — typed, deduplicated, filtered of malformed rows, and organized into dimensions, facts, and a bridge table.

**What comes in:** 5 raw bronze tables from `employeedatacatalog.bronze_movie` (all columns still strings, some rows corrupted by CSV parsing issues).

**What goes out:** 9 Delta tables in `employeedatacatalog.silver_movie` — 4 dimensions, 4 facts, 1 bridge.

---

### Dimension Tables (the "who/what/when")
| Table | Rows | Purpose |
| --- | --- | --- |
| `dim_movies` | 9,770 | Core movie attributes with proper types (int, date, boolean) |
| `dim_genres` | 19 | Genre reference lookup (genre_id, genre_name, movie_count) |
| `dim_people` | 116,478 | Deduplicated union of all cast + crew by person_id |
| `dim_date` | 6,486 | Calendar attributes derived from release dates |

### Fact Tables (the "how much/how many")
| Table | Rows | Purpose |
| --- | --- | --- |
| `fact_movie_metrics` | 9,770 | Budget, revenue, profit, ROI%, ratings, popularity |
| `fact_movie_cast` | 150,044 | Actor–movie relationships with character and billing order |
| `fact_movie_crew` | 63,632 | Crew–movie relationships with job title and department |
| `fact_movie_reviews` | 13,073 | User reviews (heavily filtered for malformed CSV rows) |

### Bridge Table
| Table | Rows | Purpose |
| --- | --- | --- |
| `bridge_movie_genres` | 22,094 | Many-to-many: exploded genres linked back to dim_genres |

---

### Steps:
1. **Read bronze tables** — Pull all 5 tables from Unity Catalog
2. **Build dimensions** — Cast types, parse dates (custom `dd/MM/yy` parser with century pivot at year 26), deduplicate people across cast/crew, generate a date dimension from all release dates
3. **Build facts + bridge** — Compute derived metrics (profit, ROI), filter malformed review rows using regex (hex ID check + numeric movie_id check), explode comma-separated genre lists into the bridge table
4. **Write to Delta** — Persist under `silver/movie/{table}` and register in Unity Catalog

### Key Technical Decisions:
- **Date parsing:** Two-digit years use a pivot — `yy > 26 → 1900s`, `yy ≤ 26 → 2000s` (covers 1927–2026)
- **Malformed row filtering:** Movies require `id` to be numeric; reviews require `id` to be a hex string and `movie_id` to be numeric
- **Path isolation:** Tables written to `silver/movie/` subdirectory to avoid collisions with other projects in the same storage account

In [0]:
from pyspark.sql.functions import *

In [0]:
# Same path layout as bronze — all layers share the same ADLS root
root_path = "abfss://employee@dataanlysisazuredatalake.dfs.core.windows.net"
bronze_path = f"{root_path}/bronze"
silver_path = f"{root_path}/silver"
gold_path = f"{root_path}/gold"

# Unity Catalog schema names
bronze_sch = "bronze_movie"
silver_sch = "silver_movie"
gold_sch = "gold_movie"

movies_db = "employeedatacatalog"

In [0]:
# Pull in all five bronze tables.
# These still have raw string types and bronze metadata columns —
# we'll clean them up and cast types in the cells below.

df_movie = spark.sql("SELECT * FROM employeedatacatalog.bronze_movie.movies")
df_genres = spark.sql("SELECT * FROM employeedatacatalog.bronze_movie.genres")
df_cast = spark.sql("SELECT * FROM employeedatacatalog.bronze_movie.cast")
df_crew = spark.sql("SELECT * FROM employeedatacatalog.bronze_movie.crew")
df_reviews = spark.sql("SELECT * FROM employeedatacatalog.bronze_movie.reviews")

In [0]:
# ---------------------------------------------------------------
# Date parsing note:
# The movies CSV stores release_date as dd/MM/yy (e.g. "25/05/77").
# Two-digit years are ambiguous, so we use a pivot at 26 (current year):
#   yy > 26  → 1900s  (77 → 1977, 41 → 1941)
#   yy <= 26 → 2000s  (03 → 2003, 26 → 2026)
# This covers everything from Metropolis (1927) to recent films.
# ---------------------------------------------------------------

def parse_ddMMyy(col_name):
    parts = split(col(col_name), "/")
    dd, mm = parts[0], parts[1]
    yy = parts[2].cast("int")
    yyyy = when(yy > 26, yy + 1900).otherwise(yy + 2000)
    return when(
        col(col_name).isNotNull() & (col(col_name) != "") & col(col_name).contains("/"),
        to_date(concat_ws("-", yyyy.cast("string"), lpad(mm, 2, "0"), lpad(dd, 2, "0")), "yyyy-MM-dd")
    )

# Some bronze rows are junk — the CSV parser split review content across
# movie columns, giving us non-numeric IDs like "Well" or "From the time...".
# Filter those out here so every downstream table starts clean.
df_movie_valid = (
    df_movie
    .filter(col("_bronze_is_valid") == True)
    .filter(col("id").rlike("^\\d+$"))
)

# ── dim_movies ──────────────────────────────────────────
# Core movie attributes with proper types. Keeps genre_list as a
# comma-separated string for quick reference; the normalized version
# lives in bridge_movie_genres.
dim_movies = (
    df_movie_valid
    .select(
        col("id").cast("int").alias("movie_id"),
        trim(col("title")).alias("title"),
        trim(col("original_title")).alias("original_title"),
        trim(col("overview")).alias("overview"),
        parse_ddMMyy("release_date").alias("release_date"),
        col("runtime").cast("int").alias("runtime_minutes"),
        col("status"),
        trim(col("tagline")).alias("tagline"),
        trim(col("homepage")).alias("homepage"),
        col("original_language"),
        col("poster_path"),
        col("backdrop_path"),
        when(col("adult") == "1", lit(True)).otherwise(lit(False)).alias("is_adult"),
        col("genres").alias("genre_list"),
        when(col("updated_at").isNotNull() & (col("updated_at") != ""),
             to_timestamp(col("updated_at"))).alias("last_updated_at")
    )
    .dropDuplicates(["movie_id"])
)

# ── dim_genres ──────────────────────────────────────────
# Straight from the genres reference table. 19 genres total.
dim_genres = (
    df_genres
    .filter(col("_bronze_is_valid") == True)
    .select(
        col("id").alias("genre_id"),
        trim(col("name")).alias("genre_name"),
        col("movie_count")
    )
    .dropDuplicates(["genre_id"])
)

# ── dim_people ──────────────────────────────────────────
# Merge cast and crew into one people dimension, deduped on person_id.
# A person who acts AND directs shows up once.
people_cast = df_cast.filter(col("_bronze_is_valid")).select("person_id", "name")
people_crew = df_crew.filter(col("_bronze_is_valid")).select("person_id", "name")

dim_people = (
    people_cast.unionByName(people_crew)
    .withColumn("person_name", trim(col("name")))
    .drop("name")
    .dropDuplicates(["person_id"])
)

# ── dim_date ────────────────────────────────────────────
# One row per unique release date. Useful for time-based joins
# and dashboard filters (year, quarter, month, day of week).
dim_date = (
    dim_movies
    .select("release_date")
    .filter(col("release_date").isNotNull())
    .dropDuplicates()
    .select(
        date_format("release_date", "yyyyMMdd").cast("int").alias("date_key"),
        col("release_date").alias("full_date"),
        year("release_date").alias("year"),
        month("release_date").alias("month"),
        dayofmonth("release_date").alias("day"),
        quarter("release_date").alias("quarter"),
        dayofweek("release_date").alias("day_of_week"),
        date_format("release_date", "EEEE").alias("day_name"),
        date_format("release_date", "MMMM").alias("month_name")
    )
)

for name, df in [("dim_movies", dim_movies), ("dim_genres", dim_genres),
                 ("dim_people", dim_people), ("dim_date", dim_date)]:
    print(f"{name}: {df.count():,} rows")

In [0]:
# ── fact_movie_metrics ──────────────────────────────────
# Financial and rating numbers per movie. Profit and ROI are
# computed here so downstream queries don't have to repeat the math.
# ROI is null when budget is zero (avoids divide-by-zero).
fact_movie_metrics = (
    df_movie_valid
    .select(
        col("id").cast("int").alias("movie_id"),
        col("budget").cast("long").alias("budget"),
        col("revenue").cast("long").alias("revenue"),
        (col("revenue").cast("long") - col("budget").cast("long")).alias("profit"),
        when(
            col("budget").cast("long") > 0,
            round((col("revenue").cast("long") - col("budget").cast("long")) / col("budget").cast("long") * 100, 2)
        ).alias("roi_pct"),
        col("vote_average").cast("double").alias("vote_average"),
        col("vote_count").cast("int").alias("vote_count"),
        col("popularity").cast("double").alias("popularity")
    )
    .dropDuplicates(["movie_id"])
)

# ── fact_movie_cast ─────────────────────────────────────
# Which actors played which roles. cast_order = 0 means top billing.
fact_movie_cast = (
    df_cast
    .filter(col("_bronze_is_valid") == True)
    .select(
        col("id").alias("cast_id"),
        col("movie_id"),
        col("person_id"),
        trim(col("character")).alias("character"),
        col("cast_order").cast("int").alias("cast_order")
    )
    .dropDuplicates(["cast_id"])
)

# ── fact_movie_crew ─────────────────────────────────────
# Behind-the-scenes roles: directors, writers, producers, etc.
fact_movie_crew = (
    df_crew
    .filter(col("_bronze_is_valid") == True)
    .select(
        col("id").alias("crew_id"),
        col("movie_id"),
        col("person_id"),
        trim(col("job")).alias("job"),
        trim(col("department")).alias("department")
    )
    .dropDuplicates(["crew_id"])
)

# ── fact_movie_reviews ──────────────────────────────────
# The reviews CSV was badly parsed — commas in review text spilled
# content into the wrong columns. We keep only rows where:
#   - id looks like a valid TMDB hex string (24+ chars)
#   - movie_id is purely numeric
# This drops the garbled rows while keeping real reviews.
fact_movie_reviews = (
    df_reviews
    .filter(col("_bronze_is_valid") == True)
    .filter(col("id").rlike("^[a-f0-9]{20,}$"))
    .filter(col("movie_id").rlike("^\\d+$"))
    .select(
        col("id").alias("review_id"),
        col("movie_id").cast("int").alias("movie_id"),
        trim(col("author")).alias("author"),
        trim(col("author_username")).alias("author_username"),
        col("author_rating").cast("double").alias("author_rating"),
        trim(col("content")).alias("content"),
        to_timestamp(col("created_at")).alias("created_at"),
        to_timestamp(col("updated_at")).alias("updated_at"),
        col("content_length").cast("int").alias("content_length")
    )
    .dropDuplicates(["review_id"])
)

# ── bridge_movie_genres ─────────────────────────────────
# Movies have a comma-separated genres string ("Drama,War").
# We explode it into one row per movie-genre pair, then join
# with dim_genres to pick up the genre_id for star-schema joins.
bridge_movie_genres = (
    df_movie_valid
    .filter(col("genres").isNotNull() & (col("genres") != ""))
    .select(
        col("id").cast("int").alias("movie_id"),
        explode(split(col("genres"), ",")).alias("genre_name")
    )
    .withColumn("genre_name", trim(col("genre_name")))
    .filter(col("genre_name") != "")
    .join(
        dim_genres.select("genre_id", "genre_name"),
        on="genre_name",
        how="left"
    )
    .select("movie_id", "genre_id", "genre_name")
    .dropDuplicates(["movie_id", "genre_name"])
)

for name, df in [("fact_movie_metrics", fact_movie_metrics), ("fact_movie_cast", fact_movie_cast),
                 ("fact_movie_crew", fact_movie_crew), ("fact_movie_reviews", fact_movie_reviews),
                 ("bridge_movie_genres", bridge_movie_genres)]:
    print(f"{name}: {df.count():,} rows")

In [0]:
silver_tables = {
    "dim_movies": dim_movies,
    "dim_genres": dim_genres,
    "dim_people": dim_people,
    "dim_date": dim_date,
    "fact_movie_metrics": fact_movie_metrics,
    "fact_movie_cast": fact_movie_cast,
    "fact_movie_crew": fact_movie_crew,
    "fact_movie_reviews": fact_movie_reviews,
    "bridge_movie_genres": bridge_movie_genres,
}

# Write each table under silver/movie/ to avoid path overlap
# with other projects (e.g. silver_youtube.dim_date_new)
for name, df in silver_tables.items():
    table_path = f"{silver_path}/movie/{name}"
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(table_path)
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {movies_db}.{silver_sch}.{name}
                  USING DELTA LOCATION '{table_path}'""")
    print(f"Saved {movies_db}.{silver_sch}.{name}")

print("\nSilver layer complete!")